# ASTR 457: The Midterm

Posted Thu Oct 8, due Wed Oct 14 by Noon (fork → PR, as always). Graded like a lab.

Three parts. Each is designed to test something every lab so far has come back to - which error bar is the right one, whether the model needs the extra parameter, and whether the check you ran could have failed. You've now got the tools to do all of this, so let's do it on numbers you haven't seen: 250 real Type Ia supernovae, and someone else's analysis of real white dwarfs.

Part 1 is 30 pts (6 each), Part 2 is 45 pts, and Part 3 is 25 pts. You'll need 2b and 2c right to get 2d right.

Remember, the rules are the syllabus rules - open book, you can use AI tools (document them in 3b), you can work together and discuss problems and solutions, but you have to write up your own solution and run it on **your own data** in `data/<your netid>/` (nobody else's numbers will match yours).

Put your notebook in `submissions/<your netid>/midterm.ipynb`. The path below (`DATA`) reaches your files from there. Make sure it runs top to bottom from a fresh kernel ("Restart & Run All") - that is how it will be graded, and how it will be run if you defend it.

Throughout, make sure every number you quote has a unit, an error bar and a sensible number of digits (two significant figures on the error bar, and round the value to match).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize, brentq
from scipy.interpolate import interp1d
import corner
from numpy.polynomial.polynomial import Polynomial
import astropy.cosmology
import emcee


NETID = 'asteves2'   # <-- set this
DATA = f'../../labs/midterm/data/{NETID}/'   # from submissions/<your netid>/

def load(name):
    """Read one of your CSV files as a pandas table; d['column'].to_numpy() gives a numpy array. Names stay strings."""
    return pd.read_csv(DATA + name + '.csv', dtype={'name': str})

## Part 1: Error bars (30 pts, 6 each)

Five short scenarios. Each one is a number someone handed you - say whether it's right, compute the right one, and say in a sentence or two why. Show the code. Each of these should be about 20 minutes of work.

### 1a - The extinction coefficient (`p1a_extinction.csv`)

An observer measured a photometric zero point at 15 different airmasses through one night. Extinction in magnitudes grows in proportion to the airmass $X$, so the zero point should follow a line, $m = k\,X + m_0$, where $k$ is the extinction coefficient in mag per airmass. Columns: `airmass`, `zeropoint_mag`, `sigma_quoted_mag` (the pipeline's error on each point).

1. Fit the line by weighted least squares with the quoted errors. Report $k$ and its error bar computed from the quoted errors alone.
2. Now run `np.polyfit(airmass, zeropoint_mag, 1, w=1/sigma_quoted_mag, cov=True)` and report its error bar on $k$. Explain exactly what `polyfit` did to get a different number (read its documentation for `cov`).
3. Compute $\chi^2$ and the number of degrees of freedom. How probable is a $\chi^2$ this large or this small if the quoted errors were right? (Compute it from the $\chi^2$ distribution.)
4. Report the error bar on $k$ you'd quote, and justify it. What does your answer to (3) tell you about the pipeline's error bars?

In [ ]:
# 1a: your work here

#Loading data for extinction events
name1a = 'p1a_extinction'
dat1a = load(name1a)

X, m, sig = dat1a['airmass'], dat1a['zeropoint_mag'], dat1a['sigma_quoted_mag']
p = np.polyfit(X, m, 1, w = 1 / sig, cov = 'unscaled')
sig_k = np.sqrt(p[1][0][0] + p[1][1][1])

x = np.linspace(np.min(X) - 0.1, np.max(X) +0.1, 100)

resid = m - np.polyval(p[0], X)
dof = len(X) - 2

chi2_red = np.sum ((resid / sig)**2) / dof
z = (chi2_red - dof) / np.sqrt(2 * dof)

p_small = 1 - stats.norm.cdf(z)

fig = plt.figure()
plt.errorbar(X, m, yerr=sig, fmt='o')
plt.plot(x, p[0][1] + x * p[0][0], linestyle = '--')

plt.xlabel('Airmass')
plt.ylabel('Zero point')

plt.show()

print(f'Reduced chi squared: {chi2_red:.3f}')
print(f'Degrees of freedom: {dof:d}')
print(f'Z score: {z:.3f}')
print(f'Probability of Z: {p_small:.3f}')

**ANSWER:**

The fit produces an extinction coefficient $k = $ 0.240 $\pm$ 0.020. Because the probability of obtaining the calculated Z-score, it is likely that the errorbars from the pipeline are underreported.

### 1b - The centroid (`p1b_centroid.csv`)

A classmate measured the position of a star from `n_photons` photons, each landing at a position drawn from a Gaussian of width `psf_sigma_arcsec` around the true position. They report an error on the position of `quoted_sigma_x0_arcsec`, from a clever weighting scheme they found online.

1. Compute the Cramér-Rao bound on the error of any unbiased estimator of the position (Lab 02).
2. Is the quoted error possible? If not, say what the classmate's number must mean instead, and name two concrete mistakes that would produce an error bar this small.

In [ ]:
# 1b: your work here
name1b = 'p1b_centroid'
dat1b = load(name1b)

psf_sig, sig, N = dat1b['psf_sigma_arcsec'].iloc[0], dat1b['quoted_sigma_x0_arcsec'].iloc[0], dat1b['n_photons'].iloc[0]

I = N / psf_sig**2
V = 1 / np.sqrt(I)

print(f'Quoted error: {sig:.3f}')
print(f'Cramer-Rao bound: {V:.3f}')

**ANSWER:**

The quoted error is not possible for an unbiased estimator because it is less than the Cramer-Rao bound. Two sources of error could be a PSF that is too small, or overcounting the number of photons in the experiment.

### 1c - The transit depth's error bar (`p1c_profile.csv`)

A transit-depth fit produced this profile: columns `depth_ppt` (the depth, in parts per thousand) and `minus2lnL` ($-2\ln L$ at that depth, other parameters fixed at their best values). A classmate took the second derivative of `minus2lnL` at its minimum and reported $\sigma = 1/\sqrt{d^2(-2\ln L)/d\,\textrm{depth}^2}$.

1. Find the best depth and the correct $1\sigma$ error from the curvature. Is the classmate's error bar right? If not, by what factor is it off, and why?
2. Read the $\Delta(-2\ln L) = 1$ interval directly off the profile. Is it symmetric? Compare it with your curvature answer.
3. If the classmate quotes $\pm$ their $\sigma$ on many independent datasets, what fraction of those intervals contain the true depth? Predict it, then check it with a simulation (a few thousand fake best-fit depths drawn around a truth you choose, using your correct $\sigma$).

In [ ]:
# 1c: your work here
name1c = 'p1c_profile'
dat1c = load(name1c)
depth, _2lnL = dat1c['depth_ppt'], dat1c['minus2lnL']

f = interp1d(depth, _2lnL, kind='cubic')
x0 = depth[np.argmin(_2lnL)]

res = minimize(lambda m: f(m[0]), x0 = [x0])
best, h = res.x[0], 1e-4
curve_rep = (f(best + h) - 2 * f(best) + f(best - h)) / h**2
sig_rep = 1 / np.sqrt(curve_rep)

sig_corr = np.sqrt(2 / curve_rep)

target = f(best) + 1

rng = np.random.default_rng(42)
N = 5000
truth = best
sim_best = rng.normal(truth, sig_corr, N)
covered = np.abs(sim_best - truth) <= sig_rep

print(f'Reported sigma: {sig_rep:.3f}')
print(f'Correct sigma: {sig_corr:.3f}')
print(f'Sigma ratio: {sig_rep / sig_corr:.3f}')
print(f'Coverage: {covered.mean():.3f}')

**ANSWER:**

Prediction: If the reported sigma is actually one, 68% of datasets should contain the true depth. The ratio of sigmas is high at around 70% with a coverage of 52.8% so the reported error values are likely incorrect.



### 1d - The companion mass (`p1d_mass_samples.csv`)

These are 8000 posterior samples of the mass of an unseen companion, `M2_Msun`. Three classmates report its uncertainty as (i) the variance of the samples, (ii) the 84th percentile minus the 16th, and (iii) the standard deviation divided by $\sqrt{8000}$.

1. Report the mass as a median with its 16th-84th percentile error bars, in solar masses.
2. For each of (i)-(iii): what quantity is it actually, what are its units, and by roughly what factor is it wrong as a $1\sigma$ error bar?
3. Is a symmetric $\pm$ error bar appropriate for this posterior? If not, show why.

In [ ]:
# 1d: your work here
name1d = 'p1d_mass_samples'
dat1d = load(name1d)
m = dat1d['M2_Msun'].values

sig = np.std(m)
std_mean = np.std(m) / np.sqrt(len(m))

var = np.var(m)

lo, mid, hi = np.percentile(m, [16, 50, 84], axis=0)
err_high = hi - mid
err_low = mid - lo

r1 = var / sig
r2 = (hi - lo) / sig
r3 = std_mean / sig

fig = plt.figure()
plt.hist(m, bins = 80)
plt.axvline(np.median(m), linestyle = '--', color = 'red', label = 'median')
plt.axvline(lo, linestyle = '--', color = 'orange', label = '16th percentile')
plt.axvline(hi, linestyle = '--', color = 'purple', label = '84th percentile')

plt.legend()
plt.show()

print(f'Variance / 1 sigma: {r1:.3f} ')
print(f'Width / 1 sigma: {r2:.3f} ')
print(f'Sample mean / 1 sigma: {r3:.3f} ')
print(f'Median: {np.median(m):.3f}+{err_high:.3f}, {np.median(m):.3f}-{err_low:.3f}')

**ANSWER:**

The variance of the sample is not an uncertainty measurement that can be used for the mass of the unseen companion because it carries units of squared masses. $\frac{\sigma}{\sqrt{N}}$ is the standard deviation for the sample mean. Symmetric error bars likely are not the best choice for this dataset because the distribution is skewed to the left.

### 1e - The slope at fixed intercept (`p1e_slope_intercept_samples.csv`)

Posterior samples of a slope and an intercept from a line fit. A classmate fixed the intercept at its best value, refit only the slope, and quoted the slope's error from that 1-D fit.

1. Compute the slope's error bar the classmate got (the spread of the slope *at fixed intercept*), and the marginal one (the spread of the slope with the intercept marginalized over).
2. Compute their ratio, and the correlation coefficient $\rho$ of the samples. How does the ratio depend on $\rho$? Plot the samples and draw both error bars on the plot.
3. When is the classmate's number the right one to quote? One sentence.

In [ ]:
# 1e: your work here
name1e = 'p1e_slope_intercept_samples'
dat1e = load(name1e)

m, b = dat1e['slope'].values, dat1e['intercept'].values

cov = np.cov(m, b)
sig_m = np.sqrt(cov[0, 0])

precision = np.linalg.inv(cov)
sig_fixed = 1 / np.sqrt(precision[0, 0])

r = sig_fixed / sig_m
rho = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])

m0, b0 = np.mean(m), np.mean(b)

fig = plt.figure()
plt.scatter(b, m, s = 2, alpha = 0.2)
plt.errorbar(b0, m0, yerr=sig_m, fmt='o', color = 'C3', label = 'Marginalized', capsize = 5)
plt.errorbar(b0, m0, yerr=sig_fixed, fmt='o', color = 'C1', label = 'Fixed', capsize = 5)

plt.xlabel('Intercept')
plt.ylabel('Slope')

plt.legend()
plt.show()

print(f'Marginalized sigma: {sig_m:.3f}')
print(f'Fixed'
      f'-intercept sigma: {sig_fixed:.3f}')
print(f'Ratio: {r:.3f}')
print(f'Rho: {rho:.3f}')

**ANSWER:**

The fixed error is likely underreporting the actual error because the ratio of fixed to marginalized errors is less than one.

## Part 2: 250 supernovae and a step (45 pts)

Type Ia supernovae are standardizable candles: their peak brightness, corrected for how fast they fade and for their color, measures distance well enough to have discovered cosmic acceleration. Your file `p2_pantheon_sne.csv` holds 250 real SNe Ia from the Pantheon+ compilation (Scolnic et al. 2022, ApJ 938, 113; [Brout et al. 2022](https://arxiv.org/abs/2202.04077), ApJ 938, 110), all at redshift $0.023 < z < 0.15$. For each supernova, a light-curve fit (SALT2) gives:

- `mB`: the peak apparent magnitude in the B band, and `mB_err`;
- `x1`: the light-curve *stretch* (broader light curves are brighter), and `x1_err`;
- `c`: the *color* (redder supernovae are fainter, from dust and intrinsically), and `c_err`;
- `cov_mB_x1`, `cov_mB_c`, `cov_x1_c`: the covariances between those three, from the same light-curve fit;
- `z`: the redshift; `host_logmass`, `host_logmass_err`: $\log_{10}$ of the host galaxy's stellar mass in solar masses, and its error.

The standardized brightness is the **Tripp relation**,

$$m_B = M + \mu(z) - \alpha\,x_1 + \beta\,c + \gamma\,\Theta(\log M_\star - 10),$$

where $M$ is the absolute magnitude of a "standard" SN Ia, $\mu(z)$ the distance modulus, $\Theta$ is 1 for hosts more massive than $10^{10}\,M_\odot$ and 0 otherwise, and $\gamma$ is the **host-mass step**. Supernovae in massive hosts appear to be brighter after standardization (Kelly et al. 2010; Sullivan et al. 2010); whether the step is real, and why, matters for supernova dark-energy measurements. In this form, a step that makes supernovae in massive hosts brighter has $\gamma < 0$; the published value is about $\gamma \approx -0.06$ mag. So the question for your sample is whether you need $\gamma$ at all.

There are two more things you'll need. At these redshifts the galaxies' own motions (peculiar velocities, about 250 km/s) shift $z$, and so $\mu$; add $\sigma_{\mu,\rm pec} = \frac{5}{\ln 10}\frac{250\ {\rm km/s}}{c\,z}$ to each supernova's variance. And no supernova is a perfect standard candle: add an intrinsic scatter $\sigma_{\rm int}$, a free parameter. Use a flat $\Lambda$CDM distance modulus with $H_0 = 70$ km/s/Mpc and $\Omega_m = 0.3$ (`astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(z)`); $M$ absorbs the choice of $H_0$.

We planted between **two and four contaminants** in your file - objects whose `mB` we made fainter by more than a magnitude, so they are not standard SNe Ia any more (think of a mis-classified peculiar supernova). Their names are not in the file. Real data have a few real outliers too, so not everything far from the line is planted.

### 2a - Look at the data (5 pts)

1. Compute $\mu(z)$ and plot the **Hubble residuals** $m_B - \mu(z)$ against $x_1$, against $c$, and against `host_logmass`.
2. Which way do the $x_1$ and $c$ trends go? Is that the sign the Tripp relation expects? Two or three sentences.
3. The raw trend with host mass may not have the sign of $\gamma$. Why not? (Plot $x_1$ against host mass.)
4. Can you see any of the contaminants?
5. What units do $\alpha$ and $\beta$ have?

In [ ]:
# 2a: your work here
name2 = 'p2_pantheon_sne'
dat2 = load(name2)

mu = astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(dat2['z']).value
resid = dat2['mB'] - mu

fig, ax = plt.subplots(ncols = 3, nrows = 1, figsize = (18, 6))

ax[0].scatter(dat2['x1'], resid)
ax[0].set_ylabel('Residuals')
ax[0].set_xlabel('x1')

ax[1].scatter(dat2['c'], resid)
ax[1].set_xlabel('Color')

ax[2].scatter(dat2['host_logmass'], resid)
ax[2].set_xlabel('Log(Mass)')

plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 6))

plt.scatter(dat2['host_logmass'], dat2['x1'])
plt.xlabel('Log(mass)')
plt.ylabel('x1')

plt.show()

**ANSWER:**

The x1 and color plots have clear negative and positive trends respectively. These results are expected form the Tripp relation because $m_B - \mu(z) = M - \alpha\,x_1 + \beta\,c + \gamma\,\Theta(\log M_\star - 10)$, where $x_{1}$ is negative and c is positive. Some of the highest supernovae (residuals ~ -18) may be contaminants because they must be fainter to have a residual that high. By dimensional analysis, $\alpha$ must mave units of $mag$ $(uint\ stretch)^{-1}$, and $\beta$ is unitless because color is already the difference of magnitudes.

### 2b - The likelihood (10 pts)

The parameters are $\theta = (M, \alpha, \beta, \gamma, \sigma_{\rm int})$. Each supernova's variance is

$$\sigma_i^2 = \mathbf{v}^T C_i\,\mathbf{v} + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2, \qquad \mathbf{v} = (1, \alpha, -\beta),$$

where $C_i$ is that supernova's $3\times3$ covariance of $(m_B, x_1, c)$.

1. Write out $\mathbf{v}^T C_i\,\mathbf{v}$ term by term, and explain in one sentence why the off-diagonal terms are there.
2. Write the log-likelihood. Remember the $\ln\sigma_i^2$ term - the variance depends on the parameters.
3. State a prior for every parameter, and justify each in one line.
4. Decide whether to sample $\ln\sigma_{\rm int}$ or $\sigma_{\rm int}$, and say why.
5. Decide what to do about the contaminants. You may model them or remove them, but say how you found them and what a cut would cost you.

In [ ]:
# 2b: your work here
n = len(dat2)
C = np.zeros((n, 3, 3))

C[:, 0, 0] = dat2['mB_err'].values**2
C[:, 1, 1] = dat2['x1_err'].values**2
C[:, 2, 2] = dat2['c_err'].values**2

C[:, 0, 1] = C[:, 1, 0] = dat2['cov_mB_x1'].values
C[:, 0, 2] = C[:, 2, 0] = dat2['cov_mB_c'].values
C[:, 1, 2] = C[:, 2, 1] = dat2['cov_x1_c'].values

def vTCv(alpha, beta, C):
    v = np.array([1.0, alpha, -beta])

    return v.T @ C @ v

mB = dat2['mB'].values
x1 = dat2['x1'].values
col = dat2['c'].values
c = 299792.458
Theta = (dat2['host_logmass'].values > 10).astype(float)
var_pec = ((5 / np.log(10)) * n / (c * dat2['z'].values))**2

def neg_log_likelihood(theta):
    M, alpha, beta, gamma, ln_sig = theta
    if not (-15 < ln_sig < 5):
        return 1e25
    model = M + mu - alpha*x1 + beta*col + gamma*Theta
    var = vTCv(alpha, beta, C) + var_pec + np.exp(ln_sig)**2
    r = mB - model
    return 0.5 * np.sum((r**2/var + np.log(2*np.pi*var))[keep])

M, alpha, beta, gamma, ln_sig = -19, 0.1, 2, -0.06, 0.1
theta0 = [M, alpha, beta, gamma, ln_sig]

def fit(theta0):
    res = minimize(lambda t: neg_log_likelihood(t), theta0, method = 'Nelder-Mead')
    return res.x

def std_resid(theta):
    M, alpha, beta, gamma, ln_sig = theta
    model = M + mu - alpha * x1 + beta * col + gamma * Theta
    var = vTCv(alpha, beta, C) + var_pec + np.exp(ln_sig)**2
    return (mB - model) / np.sqrt(var)

k = 3
keep = np.ones(n, dtype=bool)
theta = fit(theta0)

for _ in range(20):
    new = np.abs(std_resid(theta)) < k
    if np.array_equal(keep, new):
        break
    keep[:] = new
    theta = fit(theta0)

**ANSWER:**

Because the off-diagonal terms of the covariance matrix are non-zero, $m_{b}$, $x_{1}$ and, and $c$ are all correlated in some way. Many of the priors came from looking at the $log(mass)$ vs $x_{i}$ plots and residuals, and $\gamma=-0.06$ was quoted previously. Sig clipping was used to handle the outlier data points at a threshold of $3\sigma$, however this may come at the cost of cutting real data points and not the planted data points. I will use $ln(\sigma_{int})$ because the prior must be flat in $\sigma_{int}$, treating every magnitude as equally plausible.

### 2c - Sampling and convergence (12 pts)

1. Find the maximum a posteriori point. Then run **four independent chains** from starting points spread wider than the posterior (you choose how, and say how you chose). Any sampler is fine - your own Metropolis, `emcee` (one ensemble per chain), or `PyMC`.
2. Report $\hat R$ and the effective sample size for every parameter (`arviz.rhat`, `arviz.ess`, or `arviz.summary`), and say what values would have made you run longer.
3. Make a corner plot, and quote $\alpha$, $\beta$, $\gamma$ and $\sigma_{\rm int}$ as medians with 16th-84th percentile error bars.
4. Which parameters are correlated strongly? Comment on why. Then fix $M$ at its MAP value, refit $\gamma$, and compare that $\sigma_\gamma$ with the marginal one (cf. 1e).

`ArviZ` 1.x (install it with `conda install -c conda-forge pymc arviz`, as in `INSTALL.txt`) takes your chains as a dictionary of arrays shaped (chain, draw): `idata = az.from_dict({'posterior': {'alpha': a, 'beta': b}})`, then `az.summary(idata)`.

In [ ]:
# 2c: your work here
M_lo, M_hi = -21.0, -17.0
a_lo, a_hi = -0.5, 0.5
b_lo, b_hi = -2.0, 8.0
g_lo, g_hi = -0.5, 0.5
ls_lo, ls_hi = np.log(0.01), np.log(1.0)

def log_prior(theta):
    M, alpha, beta, gamma, ln_sig = theta
    inside = (M_lo < M < M_hi and a_lo < alpha < a_hi and b_lo < beta < b_hi
              and g_lo < gamma < g_hi and ls_lo < ln_sig < ls_hi)
    return 0.0 if inside else -np.inf

def log_post(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    val = neg_log_likelihood(theta)
    return -val if np.isfinite(val) else -np.inf

def nlp(theta):
    if not np.isfinite(log_prior(theta)):
        return 1e25
    val = neg_log_likelihood(theta)
    return -val if np.isfinite(val) else 1e25

nwalker, ndim = 64, 5
rng = np.random.RandomState(0)
lo = np.array([M_lo, a_lo, b_lo, g_lo, ls_lo])
hi = np.array([M_hi, a_hi, b_hi, g_hi, ls_hi])

def start(center, nwalkers, spread = 0.01):
    cent = center + spread * (hi - lo) * rng.normal(size = (nwalkers, ndim))
    return np.clip(cent, lo + 1e-6, hi - 1e-6)

centers = [rng.uniform(lo, hi) for _ in range(4)]

best = None
for c in centers:
    res = minimize(nlp, c, method = 'Nelder-Mead', options = {'maxiter': 20000, 'maxfev': 20000, 'xatol': 1e-6, 'fatol': 1e-6})
    if best is None or res.fun < best.fun:
        best = res
    theta_ml = best.x

centers_start = [np.clip(theta_ml + 0.05 * (hi - lo) * rng.normal(size = ndim), lo + 1e-6, hi + 1e-6,) for _ in range(4)]

chains, taus = [], []

for c in (centers_start):
    ens = start(c, nwalker)
    sampler = emcee.EnsembleSampler(nwalker, ndim, log_post)
    sampler.run_mcmc(ens, 10000)
    taus.append(sampler.get_autocorr_time(tol = 0))
    chains.append(sampler)

In [ ]:
tau = np.max(taus)
burn = int(5 * tau)
thin = max(1, int(0.5 * tau))

def gelman_rubin(chains):
    m, n = chains.shape
    B = n / (m - 1) * np.sum((chains.mean(axis=1) - chains.mean())**2)
    W = np.mean(chains.var(axis=1, ddof=1))
    V = (n - 1) / n * W + B / n
    return np.sqrt(V / W)

rhat = []
for j in range(ndim):
    x = np.stack([s.get_chain(discard = burn, flat = True)[:, j] for s in chains])
    rhat.append(gelman_rubin(x))

tau_post = np.array([s.get_autocorr_time(discard = burn, tol = 0) for s in chains])
n_steps = chains[0].get_chain(discard = burn).shape[0]
ess_total = (n_steps * nwalker / tau_post).sum(axis = 0)

names = ['M', 'alpha', 'beta', 'gamma', 'ln_sig']
df = pd.DataFrame({'R_hat': rhat}, index = names)
df['tau (steps)'] = tau_post.max(axis=0)
df['ESS total'] = ess_total

df

In [ ]:
samples = np.concatenate([s.get_chain(discard = burn, thin = thin, flat = True) for s in chains])

fig = corner.corner(samples, labels = names, quantiles = [0.16, 0.5, 0.84], show_titles = True)

In [ ]:
def nlp(theta):
    lp = log_post(theta)
    return -lp if np.isfinite(lp) else 1e25

res = minimize(nlp, theta0, method = 'Nelder-Mead')
res = minimize(nlp, res.x, method = 'Nelder-Mead')
M_map = res.x[0]

def nll_fixM(tht):
    a, b, g, s = tht
    return nlp(([M_map, a, b, g, s]))

tht0 = res.x[1:]
res_fix = minimize(nll_fixM, tht0, method = 'Nelder-Mead')
res_fix = minimize(nll_fixM, res_fix.x, method = 'Nelder-Mead')

cov = np.cov(samples, rowvar = False)
sig_g_marg = np.sqrt(cov[3, 3])
P = np.linalg.inv(cov)
cond = np.linalg.inv(P[1:, 1:])
sig_g_fixedM = np.sqrt(cond[2, 2])

print(f'Marginalized sigma: {sig_g_marg:.3f}')
print(f'Fixed M sigma: {sig_g_fixedM:.3f}')
print(f'Ratio: {sig_g_fixedM / sig_g_marg:.3f}')

 **ANSWER:**

From the corner plots, $\alpha = 0.13 \pm0.01$, $\beta = 2.78^{+0.14}_{-0.15}$, $\gamma = -0.04 \pm 0.03$, and $log(\sigma_{int}) = -1.74 \pm 0.05$. $\gamma$ and $M$ have a strong negative correlation because $\Theta$ is 0 for lower mass stars and 1 for higher mass stars so raising $\gamma$ only brightens high mass stars. $\gamma$ and $\alpha$ have a noticeable negative correlation for the same reason. $\beta$ amd $M$ seem to hae a weak positive correlation because increasing the color magnitude of $c$ makes the star dimmer, producing a higher $M$. There doesn't seem to be correlations between the other parameters. From the ratio of $\sigma_{gamma, fixed M}$ to $\sigma_{\gamma, marginalized}$, $\gamma$ and $M$ are correlated because $R = 0.650$.

### 2d - Model comparison (10 pts)

Compare $M_0$ ($\gamma = 0$) with $M_1$ ($\gamma$ free).

1. Compute $2\ln B_{1,0}$ with the Laplace approximation (Day 13), with a prior on $\gamma$ you state and defend. How does it change if that prior is twice as wide?
2. Compute $\textrm{BIC}_0 - \textrm{BIC}_1$ for comparison.
3. How strong is your answer on Day 8's scale (0-2 weak, 2-6 positive, 6-10 strong, above 10 very strong)? Compare $\gamma$ with the published $-0.06$ mag. Could your 250 supernovae have detected a step that size? (One way: your $\sigma_\gamma$ tells you how many $\sigma$ a $-0.06$ step would be.)

In [ ]:
# 2d: your work here
def fit(f, x0):
    r = minimize(f, x0)
    r = minimize(f, r.x)
    return r

res1 = fit(neg_log_likelihood, theta0)

def nllg0(tht):
    M, a, b, log_sigma = tht
    return neg_log_likelihood([M, a, b, 0.0, log_sigma])

res0 = minimize(nllg0, res1.x[[0, 1, 2, 4]])

C1, C0 = res1.hess_inv, res0.hess_inv

def ln_Z(res, C, ln_prior):
    d = len(res.x)
    sign, logDetC = np.linalg.slogdet(C)
    return -res.fun + ln_prior + 0.5 * d * np.log(2 * np.pi) + 0.5 * logDetC

#Width of parameter ranges
w_M, w_a, w_b, = M_hi - M_lo, a_hi - a_lo, b_hi - b_lo
w_ls = ls_hi - ls_lo
w_g = g_hi - g_lo

lnp0 = -np.log(w_M * w_a * w_b * w_ls)    #M0 (gamma = 0)
lnp1 = lnp0 - np.log(w_g)                 #M1 (gamma free)

def two_lnB(w_gamma):
    lp1 = lnp0 - np.log(w_gamma)
    return 2 * (ln_Z(res1, C1, lp1) - ln_Z(res0, C0, lnp0))

N = int(keep.sum())

bic0 = 2 * res0.fun + 4 * np.log(N)
bic1 = 2 * res1.fun + 5 * np.log(N)

print(f'2lnB_0 = {lnp0:.3f}')
print(f'2lnB_1 = {lnp1:.3f}')

print(f'BIC0 = {bic0:.3f}, BIC1 = {bic1:.3f}')
print(f'BIC0 - BIC1 = {np.abs(bic0 - bic1):.3f}')


**ANSWER:**

Using the Laplace approximation, it does not appear that doubling the prior has an effect. The ratio of $\gamma = 0$ to $\gamma$ free is 5.443, which suggests a positive against the host-mass step.

### 2e - Testing your pipeline (8 pts)

Design one test of your pipeline that would come out differently if your pipeline were wrong, and run it. State **before** you run it what result would count as a failure. One suggestion: generate fake supernovae at your redshifts, stretches and colors from parameters you choose (including a $\gamma$ you choose), fit 30 or more such datasets (maximum-likelihood fits with curvature error bars are fine here; you don't need MCMC for each), and check how often your 68% intervals contain the truth.

In [ ]:
# 2e: your work here
n = len(dat2)
rng = np.random.default_rng(42)
truth = np.array([-19.3, 0.14, 3.1, -0.06, np.log(0.10)])
L = np.linalg.cholesky(C)

def make_fake_set(truth):
    M, a, b, g, ls = truth
    z = rng.standard_normal((n, 3))
    noise = (L @ z[:, :, None])[:, :, 0]
    x1_obs, c_obs = x1 + noise[:, 1], col + noise[:, 2]
    extra = rng.normal(0, np.sqrt(var_pec + np.exp(ls)**2))
    mB_obs = M + mu - a * x1 + b * col + g * Theta + noise[:, 0] + extra
    return mB_obs, x1_obs, c_obs

def nll_data(theta, mBd, x1d, cd):
    M, a, b, g, ls = theta
    if not (-15 < ls < 5):
        return 1e25
    var = vTCv(a, b, C) + var_pec + np.exp(ls)**2
    r = mBd - (M + mu - a * x1d + b * cd + g * Theta)
    return 0.5 * np.sum(r**2 / var + np.log(2 * np.pi * var))

def fit_bfgs(f, x0):
    r = minimize(f, x0, method = 'BFGS')
    return minimize(f, r.x, method = 'BFGS')

def num_hess(f, x, h):
    k = len(x)
    H = np.zeros((k, k))
    for i in range(k):
        for j in range(i, k):
            ei = np.zeros(k)
            ei[i] = h[i]
            ej = np.zeros(k)
            ej[j] = h[j]
            H[i, j] = H[j, i] = (f(x + ei + ej) - f(x + ei - ej) - f(x - ei + ej) + f(x - ei - ej)) / (4 * h[i] * h[j])
    return H

x_start = truth + np.array([0.05, 0.02, 0.2, 0.0, 0.1])
h = np.array([0.003, 0.003, 0.03, 0.005, 0.01])

est, sig = [], []
for _ in range(100):
    d = make_fake_set(truth)
    f = lambda t: nll_data(t, *d)
    r = fit_bfgs(f, x_start)
    H = num_hess(f, r.x, h)
    est.append(r.x)
    sig.append(np.sqrt(np.diag(np.linalg.inv(H))))

est, sig = np.array(est), np.array(sig)

pull = (est - truth) / sig
cover = (np.abs(pull) < 1).mean(axis = 0)

for n, m in enumerate(['M', 'alpha', 'beta', 'gamma', 'ln_sig']):
    print(f'{m:7s} coverage = {cover[n]:.3f}')

**ANSWER:**

To test the pipeline, I will choose generate artificial datasets using the best fit values found in part 2c. If the pipeline is correct, a $1\sigma$ error bar (68%) should contain the truth about 68% of the time. All values except for $\beta$ have a coverage of 60%-65%, which is within 10% of the required value to pass for coverage. The error on $\beta$ being far smaller than expected is likely due to the fact that noise is added to the color value, but then is treated as if it is exact in the fit.

## Part 3: The referee report (25 pts)

White dwarfs are the cores of dead stars like the Sun. Your file `p3_wd_masses.csv` holds 600 real DA (hydrogen-atmosphere) white dwarfs from the 100 pc sample of [Kilic et al. (2020)](https://arxiv.org/abs/2006.00323), ApJ 898, 84: `name`, `teff_K`, `mass_Msun`, `mass_err_Msun`. Most single white dwarfs have masses near $0.6\,M_\odot$; Kilic et al. report a narrow peak there plus an excess of more massive ones, around $0.7$-$0.9\,M_\odot$, which merger products alone do not explain.

The notebook `labs/midterm/ai_solution.ipynb` fits your masses with a two-population model using `emcee`. Run it from that directory, after setting `NETID` in its first cell. It runs top to bottom and the plots look nice, but it's wrong in at least three substantive ways - and if you don't look at your data, you'll fool yourself the same way.

### 3a - The report (20 pts)

1. **Identify** at least three substantive errors (not style; "it used 32 walkers and I'd use 48" is not one).
2. **Demonstrate** each on *your* file: show, with a number or a plot, what the error does to the answer. A demonstration can come out negative: if an error doesn't move your numbers, show that, and say what it would take for it to matter.
3. **State** the correct treatment, and run it where you can.

In [ ]:
# 3a: your work here

NETID = 'asteves2'   # <-- set this
d = np.genfromtxt(f'/Users/andrewsteves/ast457_2026_Fall/labs/midterm/data/{NETID}/p3_wd_masses.csv', delimiter=',', names=True, dtype=None, encoding='utf-8')
mass = d['mass_Msun']

med, sd = np.median(mass), np.std(mass)
keep = np.abs(mass - med) < 3 * sd
m = mass[keep]

#Error 1: Model convergence
def log_prob_ai(p, m):
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1):
        return -np.inf
    return np.sum(np.log(f * stats.norm.pdf(m, mu1, s1) + (1 - f) * stats.norm.pdf(m, mu2, s2)))

# starting guess from a quick split of the sample at 0.7 Msun
lo, hi = m[m < 0.7], m[m >= 0.7]
p0 = np.array([lo.mean(), lo.std(), hi.mean(), hi.std(), lo.size / m.size])

def conv(num_steps):
    np.random.seed(42)
    nwalk, ndim = 32, 5
    start = p0 + 1e-3 * np.random.randn(nwalk, ndim)
    sampler = emcee.EnsembleSampler(nwalk, ndim, log_prob_ai, args=(m,))
    sampler.run_mcmc(start, num_steps, progress=False)
    chain = sampler.get_chain(discard=500)              # (steps, walkers, params)

    return chain

convg = conv(num_steps = 10000)

def gelman_rubin(x):                                  # x: (steps, chains)
    n, mchains = x.shape
    B = n * np.var(x.mean(axis=0), ddof=1)
    W = np.mean(np.var(x, axis=0, ddof=1))
    return np.sqrt(((n - 1) / n * W + B / n) / W)

res = {}
names = ['mu1', 'sigma1', 'mu2', 'sigma2', 'f']
for i, nm in enumerate(names):
    print(f"R-hat({nm}) = {gelman_rubin(convg[:, :, i]):.3f}")


fig, axs = plt.subplots(5, 1, figsize=(8, 8), sharex=True)
for i, nm in enumerate(names):
    axs[i].plot(convg[:, :, i], lw=0.3, alpha=0.5); axs[i].set_ylabel(nm)
axs[-1].set_xlabel('step'); plt.tight_layout(); plt.show()

In [ ]:
#Error 2: Incorrect treatment of clipping masses
T, mass_err_Msun = d['teff_K'], d['mass_err_Msun']

def fit_ai(dat):
    np.random.seed(42)
    nwalk, ndim = 32, 5
    start = p0 + 1e-3 * np.random.randn(nwalk, ndim)
    sampler = emcee.EnsembleSampler(nwalk, ndim, log_prob_ai, args=(dat,))
    sampler.run_mcmc(start, 5000)
    return sampler.get_chain(discard = 200, flat = True)

flat_clip = fit_ai(m)
flat_unclipped = fit_ai(mass)

names = ['mu1', 'sigma1', 'mu2', 'sigma2', 'f']
for i, name in enumerate(names):
    for lab, fl in (('clipped', flat_clip), ('unclipped', flat_unclipped)):
        lo, mid, hi = np.percentile(fl[:, i], [16, 50, 84])
        print(f'{name:7s}| {lab:8s} {mid:.4f} +{hi-mid:.4f} -{mid-lo:.4f}')
        print('------------------------------------------------------')


In [ ]:
#Error 3: incorrect error statement
best = np.median(flat_clip, axis = 0)
sig_mu1 = flat_clip[:, 0].std()

for i, nm in enumerate(names):
    lo, mid, hi = np.percentile(flat_clip[:, i], [16, 50, 84])
    print(f'{nm:7s} {mid:.4f} +{hi-mid:.4f} -{mid-lo:.4f}')

**REFEREE REPORT:**

The first error I noticed is that the AI falsely stated that all models had converged because they all had $\hat{R}$ values of less than 1.1 which is untrue by inspecting the actual values. This is likely due to there not being enough steps in the MCMC simulation. After increasing the steps from 3000 to 10000, all parameters have a much more reasonable value of around 1, and more visual certainty of convergence. The second error seems to come from the clipping that was applied to remove outliers. In the way that it was done, only high mass stars (> $1.1M_\odot$) are clipped, where the lowest mass star is $0.22M_\odot$ and the lower bound for clipping is $0.13M_\odot$. From the plots, most of the highest mass stars have normal errors, which suggests less random noise. The correct treatment for this is to not clip the data, at least with using the masses of the white dwarfs. Using the clipped vs unclipped data makes very little difference for the posteriors of the model. The third error is the stated error on the population means. The quoted value is the spread of the population. Shown above are the 16-84 percentile error bars, which say that the distribution is not symmetric and that the reported errors are incorrect.

### 3b - AI-use appendix (5 pts)

1. Which tools did you use, for what, what did they get wrong, and how did you catch it? "I used ChatGPT to help" is useless; "none" is fine if that's true. I promise I don't care which - just be honest about what you did.
2. What would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing, in two or three sentences.

**AI-USE APPENDIX:**

AI was used predominantly to help with understanding the questions, providing context when needed, and locating sections in the lectures that helped with any given section. I had Claude write the posterior and nlp functions because the signs of each were causing errors in the MCMC and minimize functions. It also helped with general debugging when errors I could not trace arose. One error that could have occured in due to the AI's output was the correct of the posterior funciton(s). If the sign were incorrect the minimize funciton would send values to $-\infty$ becuase the minimum is unbounded. I worked around this by having posterior functions for each sign. This is not the most elegant solution, but it works.

Get started. You know how to do all of this. If not, there's something in a lab or class notebook that can help. Slack me with questions. Best of luck.